# RouteHunter

**The app is static.** A CSV file is the single source of truth for the dataset — there is no Contribute/admin pipeline and no in-app way to modify the data. Each CSV you point this at is effectively a version of the app's dataset. To add or correct data, edit the CSV directly and re-run this notebook.

The only thing that happens at runtime beyond reading the CSV is an optional, session-only cache of CASP-predicted routes (Section 4) — it lives in memory for this run and is never written back anywhere.

In [1]:
from routehunter import RouteHunterApp

# This single call is the entire "startup" of the app: load the CSV,
# build the in-memory dataset, done.
app = RouteHunterApp.from_csv("data/routehunter_seed.csv")

print(app.load_report.summary())

[14:59:51] Invalid InChI prefix in generating InChI Key
[14:59:51] Invalid InChI prefix in generating InChI Key


Rows processed        : 1392
Loaded (new targets)  : 1278
Loaded (new routes)   : 112
Skipped (duplicates)  : 0
Errors                : 2

First few errors:
  row 348: Could not derive InChIKey for: 'CC1C=C(C)C(NC(CN2CCN(C3CC(CC4C=CC=CC=4)N(C(C4C=C(C(F)(F)F)C=C(C(F)(F)F)C=4)=O)CC3)CC2)=O)=C([*])C=1'
  row 517: Could not derive InChIKey for: 'CC(OC(NC([*])CC(C1C=CC=CC=1)=O)=O)(C)C'


## Review

In [2]:
print(app.introduction())

RouteHunter -- synthesis route reference lookup (static dataset)

  Search      : give a SMILES, get papers reporting a synthesis route to it.
  
  Browse      : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route.
                
  Predict     : predict a route computationally for a target with no
                known literature synthesis (cached for this session only).
                
  Download    : export data for AI/ML training.



Current dataset:
  Targets                  : 1278
  Papers                   : 1171
  Targets with >1 paper    : 92
  Cached CASP routes       : 0
  Papers by journal:
    Organic Process Research & Development   1171
  Papers by source:
    seed                                     1171


## Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

In [4]:
result = app.search("CC(C)Cc1ccc(cc1)C(C)C(=O)O")
print(result.message)
for p in result.papers:
    print(f" - [paper] {p.title} ({p.journal}, {p.year}) doi:{p.doi}")
for r in result.casp_routes:
    print(f" - [casp:{r.engine_name}] {r.route.n_steps}-step predicted route, score={r.score}")

No synthesis route found for this structure.


## Browse

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

Uses stub `PaperFetcher`/`Classifier` here — swap in the real `MLClassifier` and a literature connector as those mature.

In [ ]:
from routehunter.browse import PaperCandidate

class DummyFetcher:
    """Stand-in for a real literature source (e.g. a PubMed connector)."""
    def fetch_recent(self, journals, n):
        return [
            PaperCandidate(
                doi=f"10.1000/dummy.{i}",
                title=f"Paper {i} from {journals[0] if journals else 'Journal'}",
                abstract="This paper describes a multi-step synthesis..." if i % 2 == 0 else "This paper is a computational study...",
                journal=journals[0] if journals else None,
                year=2026,
            )
            for i in range(n)
        ]

class DummyClassifier:
    """Stand-in for the real RouteHunter MLClassifier."""
    def predict_proba(self, title, abstract):
        return 0.9 if "multi-step synthesis" in abstract else 0.1

fetcher = DummyFetcher()
classifier = DummyClassifier()

ranked = app.hunt(fetcher, classifier, journals=["JACS"], n=6)
for p in ranked:
    print(f"{p.hunter_score:.2f}  {p.title}")


## Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [ ]:
from routehunter.core import Route, RouteStep

class DummyCASPEngine:
    """Stand-in for a real open-source CASP tool."""
    def predict_route(self, smiles):
        return Route(
            target_smiles=smiles,
            steps=[RouteStep(product_smiles=smiles, reactant_smiles=["CCBr", "O"], reaction_name="hydrolysis")],
            score=0.42,
        )

engine = DummyCASPEngine()
route = app.casp(engine, "c1ccccc1", engine_name="DummyEngine")
print(f"Predicted route ({route.n_steps} step(s), score={route.score}):")
for step in route.steps:
    print(f"  {' + '.join(step.reactant_smiles)} -> {step.product_smiles}  [{step.reaction_name}]")


In [ ]:
# The cached route now shows up in Search for the rest of this session,
# even though this molecule (benzene) has no literature route in the CSV.
result = app.search("c1ccccc1")
print(result.message)

## Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [ ]:
df = app.download()
df

## Updating the dataset

There is no save/load step — the CSV *is* the persistent state. To change the dataset:

1. Edit `data/routehunter_seed.csv` directly (add, remove, or correct rows).
2. Re-run `RouteHunterApp.from_csv(...)` — that's the entire "upgrade" to a new dataset version.

No method here writes back to the CSV; keep versioning it however you'd version any other project file (e.g. git).